In [5]:
using Random, Distributions


## Model Parameter and function setup


In [ ]:
N = 3 # Countries 
J = 10000 # Number of Goods 
sigma = 2 # Elasticity 
theta = 4 # Dispersion Parameter (Comparative Advantage)
T = ones(N, 1) * 1.5 # Technology (Absolute Advantage)
tau = ones(N, N) # Trade Cost 
L = ones(N, 1) # Labor Force 
w = ones(N, 1) # Wages 

Random.seed!(1234)

TaskLocalRNG()

In [76]:
function draw_prods(N, J, theta, T) # Draws our productivities using the CDF from the EK paper 
    U = rand(J, N)
    return (-T' ./ log.(U)).^(1/theta) 
end 

z = draw_prods(N, J, theta, T) 

100000×3 Matrix{Float64}:
 1.28804   0.982088  1.4033
 1.1399    0.926212  1.15712
 2.69918   1.60759   1.00676
 0.772789  0.949912  1.8477
 1.23099   1.00176   1.12003
 1.35348   1.03046   0.949797
 1.71153   1.00396   2.42611
 2.58855   0.94038   1.20888
 1.58776   1.81332   1.14645
 1.4264    1.22005   1.22896
 ⋮                   
 1.02283   1.30486   0.764983
 1.39426   0.894257  1.01299
 1.40556   1.29221   1.17832
 0.863864  1.43801   1.01445
 1.3017    1.20672   2.10088
 0.98758   1.05776   1.60242
 1.07787   0.963777  1.03069
 1.33437   1.22543   0.926104
 1.25674   1.33047   1.07798

In [77]:
function find_trade_shares(z, tau, w) 
    costs = zeros(J, N, N)
    for n in 1:N
        for i in 1:N
            costs[:, n, i] = tau[n, i] * w[i] ./ z[:, i]
        end 
    end 
    trade_shares = zeros(N,N)
    for n in 1:N 
        for i in 1:N
            trade_shares[n, i] = mean([argmin(costs[k, n, :]) == i for k in 1:J])
        end
    end
    return trade_shares 
end

trade_shares = find_trade_shares(z, tau, w) 


3×3 Matrix{Float64}:
 0.33427  0.33099  0.33474
 0.33427  0.33099  0.33474
 0.33427  0.33099  0.33474

In [78]:
function trade_bals(trade_shares, L, w)
    expenditure = L .* w 
    trade_matrix = trade_shares .* expenditure' # Broadcast the expenditure across the rows of trade shares 
    trade_balances = zeros(N) 
    for n in 1:N 
         exports = sum(trade_matrix[:, n])
         imports = sum(trade_matrix[n, :])
         trade_balances[n] = exports - imports
    end 
    return trade_balances
end

trade_balances = trade_bals(trade_shares, L, w)

3-element Vector{Float64}:
  0.002809999999999979
 -0.007029999999999981
  0.0042199999999998905

In [ ]:
function equilibrium_info(sigma, z, tau, L, tol = 1e-4, max_iter = 1000)
    w = ones(N)

    for iter in 1:max_iter
        trade_shares = find_trade_shares(z, tau, w)
        trade_balances = trade_bals(trade_shares, L, w)
        if abs(maximum(trade_balances)) < tol
            break 
        end
        w[2:end] .+= .01 .* trade_balances[2:end] ./ (L[2:end] .* w[2:end]) #adjust wages 2 and 3 based on imbalances in trade balances
    end
    println("Equilibrium Wages: ", w)

    costs = zeros(J, N, N)
    for n in 1:N
        for i in 1:N
            costs[:, n, i] = tau[n, i] * w[i] ./ z[:, i]
        end 
    end 
    p = [mean((minimum(costs[:, n, :], dims = 2).^(1 - sigma))).^(1 / (1-sigma)) for n in 1:N]

    println("Price Indices: ", p)

    welfare = (w .* L) ./ p

    println("Welfare: ", welfare)

    return w, welfare

end

In [ ]:
equilibrium_info(sigma, z, tau, L)

# The output from the above cell is the solution to part A

# Part B
- Symmetric Geography, $\tau = 1.1$ for all $i \neq j$

In [59]:
tau = [n == i ? 1.0 : 1.1 for n in 1:N, i in 1:N]
trade_shares_b = find_trade_shares(z, tau, w)

3×3 Matrix{Float64}:
 0.424   0.2878  0.2882
 0.2895  0.4243  0.2862
 0.2914  0.2899  0.4187

Every country now supplies a significantly larger share of their own goods as foreign productivity draws may not be able to overcome the trade cost. 

In [60]:
equilibrium_info(sigma, z, tau, L)

Equilibrium Wages: [1.0, 1.0, 1.0]
Price Indices: [0.5950680813414445, 0.5953838086420595, 0.5954934810972332]
Welfare: [1.6804799843166338; 1.6795888391402205; 1.679279508076963;;]


All that seems to change in the equilibrium is that price indices are higher and welfare is lower across the board. Wages remain unchanged.

# Part C
- Asymmetric Geography, country 3 is further away from countries 1 and 2, leading to a higher trade cost. 

In [73]:
tau = ones(N, N)
tau[:, 3] .= 1.3 
tau[3, :] .= 1.3
tau[3, 3] = 1
tau[1, 2] = 1.05
tau[2, 1] = 1.05 
tau

3×3 Matrix{Float64}:
 1.0   1.05  1.3
 1.05  1.0   1.3
 1.3   1.3   1.0

In [74]:
welfare_c, w_c = equilibrium_info(sigma, z, tau, L)

Equilibrium Wages: [1.0, 0.998249327622334, 0.9454854127590558]
Price Indices: [0.6014228458754629, 0.6015690675225642, 0.6252633134182084]
Welfare: [1.6627236674794872; 1.6594093372078023; 1.512139593782094;;]


([1.0, 0.998249327622334, 0.9454854127590558], [1.6627236674794872; 1.6594093372078023; 1.512139593782094;;])